# 11.12 - Generation & Grounding

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

The final step: use the LLM to produce an answer faithful to the retrieved context. This balances helpfulness with faithfulness - answering the question while only using the provided evidence and citing it.

## 2. Why Does This Matter?

This is where RAG produces its output. Get grounding wrong and the model drifts into parametric memory / hallucination even with perfect retrieval.

## 3. Prerequisites

Unit 11.11 (Context Construction), Phase 10 (LLMs).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Compare an llm() answer with and without grounding (same question)
- Instruct the model to use only the context and cite sources
- Spot the difference and understand the ungrounded hallucination risk

## 5. Mental Model

Generation in RAG is a well-briefed consultant: they read the evidence, synthesize, and cite. They do not add facts from memory.

```text
Context + Query -> LLM -> Answer + Citations
```


## 6. Setup
We use the shared `llm()` helper. Without a GROQ_API_KEY it returns a deterministic mock - perfect for seeing the *prompt* differences drive the answer.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Without vs With Grounding
Same question. Without grounding the model guesses from memory (hallucination risk). With grounding it must answer from the context and cite sources, forcing faithfulness even in the mock.

In [2]:
question = "What is the refund window and who covers return shipping?"

# no grounding
ungrounded = llm(f"Answer from your own knowledge: {question}")
print("NO GROUNDING:")
print("  ", ungrounded)
print()


# grounded
context = (
    "[1] faq.md p.2\nReturns are accepted within 30 days of purchase.\n\n"
    "[2] policy.pdf p.5\nThe customer pays return shipping unless the item is defective."
)
grounded = llm(
    "Answer the question using ONLY the provided context. "
    "If the context lacks the answer say you don't know. Cite sources as [1]/[2].\n"
    f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)
print("WITH GROUNDING:")
print("  ", grounded)


NO GROUNDING:
   



WITH GROUNDING:
   The refund window is **within 30 days of purchase**【1】.  
Return shipping is **paid by the customer unless the item is defective**【2】.


## 8. The Grounding Instruction Does the Work
The prompt instructs the model to (a) use only the context, (b) abstain when insufficient, (c) cite. Let's see it explicitly - this is what *you* control, not the model.

In [3]:
def grounded_prompt(question, context):
    return (
        "Answer ONLY from the context. If the context doesn't answer the question, "
        "say 'I don't have enough information'. Cite sources as [i].\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )

print(grounded_prompt(question, context))


Answer ONLY from the context. If the context doesn't answer the question, say 'I don't have enough information'. Cite sources as [i].

Context:
[1] faq.md p.2
Returns are accepted within 30 days of purchase.

[2] policy.pdf p.5
The customer pays return shipping unless the item is defective.

Question: What is the refund window and who covers return shipping?

Answer:


## 9. Hallucination Risk: Unanswerable Question
Ask something not in the context. With grounding + abstention the model should refuse; without grounding it may confidently invent a number.

In [4]:
tricky = "What is the fee for late returns after 60 days?"
print("WITHOUT grounding:", llm(f"Answer: {tricky}"))
print("WITH grounding:   ", llm(grounded_prompt(tricky, context)))


WITHOUT grounding: **Answer:**  
After 60 days past the due date, the late‑return fee is **$5.00 per day**.  

*(Note: Some libraries or rental services may have a different rate or a flat fee for very late items, so it’s always a good idea to check the specific policy of the institution you’re dealing with.)*


WITH grounding:    I don't have enough information.


## 10. Temperature: 0.0 for RAG
RAG wants deterministic, faithful answers - so temperature ~0.0. Higher temperature adds creativity but drifts from the evidence.

In [5]:
import pandas as pd
param = pd.DataFrame([
    ["Temperature", "0.0 - 0.2", "Deterministic, faithful"],
    ["Top-p", "0.9 - 1.0", "Some diversity for complex Q"],
    ["Max tokens", "500 - 1000", "Enough, not too long"],
    ["Model", "small/cheap", "Balance quality and cost"],
], columns=["Parameter", "RAG setting", "Why"])
print(param.to_string(index=False))


  Parameter RAG setting                          Why
Temperature   0.0 - 0.2      Deterministic, faithful
      Top-p   0.9 - 1.0 Some diversity for complex Q
 Max tokens  500 - 1000         Enough, not too long
      Model small/cheap     Balance quality and cost



## Common Mistakes

- Using high temperature (creative but unfaithful).
- Not instructing the model to cite sources.
- Letting the model use parametric knowledge instead of context.
- Not handling the "I don't know" case.
- Post-processing that strips citations.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Creative but wrong | High temperature | Set temperature 0.0 |
| Uses external knowledge | No context restriction | Strengthen grounding |
| Fabricated citations | Model invents references | Verify against chunks |
| Too brief | Max tokens too low | Increase max_tokens |

## Best Practices

- Use temperature 0.0 for RAG.
- Always instruct: answer only from the provided context.
- Include abstention instructions.
- Verify citations programmatically.
- Log prompt + answer for auditing.

## Hands-On Practice

1. **Basic:** Generate an answer from context.
2. **Guided:** Compare temperature 0.0 vs 0.8.
3. **Independent:** Build retrieve -> construct -> generate -> extract citations.
4. **Realistic:** Test queries where context is insufficient.
5. **Challenge:** Verify cited sources actually exist in context.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
